In [0]:
%pip install databricks_langchain langgraph langchain_core # mermaid
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip list | grep langgraph

langgraph                          1.0.10
langgraph-checkpoint               4.0.1
langgraph-prebuilt                 1.0.8
langgraph-sdk                      0.3.10


In [0]:
from databricks_langchain import ChatDatabricks
from databricks_langchain import DatabricksVectorSearch
from langchain.tools import tool
# from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent

llm = ChatDatabricks(
    endpoint="databricks-gpt-5-2",
    temperature=0 
)

@tool
def search_vector(query: str) -> str:
    """
    회사의 내부 지식 베이스나 특정 문서 데이터에서 정보를 검색할 때 사용합니다.
    질의(query)와 관련된 가장 유사한 문서의 내용을 반환합니다.
    """
    
    vector_store = DatabricksVectorSearch(
        index_name="edu260323.rag_idx.tourism_idx",
        columns=[
            'page_num', 'content_type', 'description'
        ]
    )
    
    # 검색된 결과 추출 및 결합
    result_docs = vector_store.similarity_search(query, k=3)
    docs = [f"-page_num: {doc.metadata['page_num']}\n-content_type: {doc.metadata['content_type']}\n-page_content: {doc.page_content}" for doc in result_docs]
    return "\n\n".join(docs) if docs else "관련된 정보를 찾지 못했습니다."

tools = [search_vector]


# create_react_agent는 '도구 실행 -> 결과 전달 -> 최종 답변' 루프를 자동으로 처리합니다.
# agent_executor = create_react_agent(llm, tools)
agent = create_agent(
    llm,
    tools,
    system_prompt="문서 검색을 위해 seach_vector 도구를 사용하세요",
)

query = "제주도 축제는 어디에 가장 많이 열리고 어떤 축제가 있어 ?"

# 2. 실행 (에이전트는 여러 단계를 거치므로 스트리밍이나 마지막 결과만 추출 가능)
response = agent.invoke({"messages": [("user", query)]})

# 3. 결과 확인
# response['messages']의 마지막 메시지가 LLM의 최종 답변입니다.
print(response["messages"][-1].content)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
문서 검색 결과(2024년 문화관광축제 평가 결과 분석 일부)에는 **“마을·지역별로 공연이 개최되고, 제주도 전체 지역에서 민속 경연이 분산돼 펼쳐진다”**는 식의 서술은 있지만, **“제주도 축제가 어디(어느 시/군/읍면)에 가장 많이 열린다”를 통계로 집계한 자료(지역별 개최 횟수/비중)**는 확인되지 않았습니다. 그래서 “가장 많이 열리는 곳”을 **문서 근거로 단정**하긴 어렵습니다.

다만 여행/행사 운영 관점에서 보면 축제는 보통 **접근성과 인프라가 좋은 곳**에 몰리는 경향이 있어, 제주에서는 대체로 아래 권역에 많이 열립니다.

## 축제가 많이 열리는 편인 곳(경향)
- **제주시 권역(제주시내·애월·한림 등)**: 공항/터미널 접근,

In [0]:
response = agent.invoke({"messages": [("user", "오늘 저녁은 뭘먹지?")]})

print(response["messages"][-1].content)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
오늘 저녁은 “지금 상태”에 따라 고르는 게 제일 빨라요. 아래에서 하나만 골라보세요.

- **가볍게/속 편하게**: 칼국수·잔치국수 / 죽 / 순두부찌개  
- **든든하게**: 제육볶음+밥 / 김치찌개+계란말이 / 국밥  
- **스트레스 풀기(매콤)**: 닭갈비 / 떡볶이+튀김 / 마라탕(맵기 조절)  
- **안주 겸**: 치킨+맥주 / 족발·보쌈 / 피자  
- **건강하게**: 샐러드볼+단백질(닭/연어) / 회덮밥 / 쌈밥

딱 3가지만 알려주면 더 정확히 하나로 골라줄게요:  
1) **혼밥/같이** 2) **매운 거 가능?** 3) **배달 vs 외식 vs 집밥**
